# Notebook 03 — Claude-as-Judge Evaluation (Figure 1)

Implements **Section 2.1** of the paper. Each model's reasoning trace (or final answer, for the no-thinking ablation and Llama-3.2-3B) is scored by **Claude Opus-4.7** on three criteria — *Reasoning Quality*, *Intellectual Depth*, *Reasoning Traceability* — on a 0–10 scale, plus an aggregate Overall score. Claude is used as the judge (rather than GPT) to avoid model-family bias, since GPT was used during dataset generation.

Eight models are evaluated:

| Variant | Notes |
|---|---|
| Graph-PRefLexOR (8B / 3B / 1.7B) | Structured reasoning, scored on `<think>` block |
| Qwen-8B / Qwen-1.7B | Native thinking; scored on `<think>` block |
| Qwen-8B-no-thinking / Qwen-1.7B-no-thinking | Ablation; scored on final answer only |
| Llama-3.2-3B-Instruct | No reasoning phase; scored on final answer only |

Per-model inference outputs live in `data/results/*_results*.jsonl` and are produced by the [`scripts/eval_*.py`](../scripts/) inference scripts.

## 1. Setup

In [ ]:
import os, json, re
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from dotenv import load_dotenv
from pydantic import BaseModel, Field

import anthropic

# Anthropic client — Claude Opus-4.7 is the judge (paper Ref. [35])
load_dotenv(Path("../.env"))
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL  = "claude-opus-4-7"

RESULTS_DIR = Path("../data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Evaluation schema and rubric

`EvalResult` constrains Claude's output to four numeric scores plus a one-sentence summary. `EVAL_SYSTEM_OPENENDED` encodes the full rubric: deduction tables for `[TRACE]` inputs and a structural-tier mapping for `[FINAL ANSWER ONLY]` inputs (so models without a thinking phase can still be scored fairly on traceability).

In [ ]:
class EvalResult(BaseModel):
    reasoning_quality:      float = Field(ge=0, le=10)
    intellectual_depth:     float = Field(ge=0, le=10)
    reasoning_traceability: float = Field(ge=0, le=10)
    overall_score:          float = Field(ge=0, le=10)
    summary:                str   = ""
    error:                  str   = ""

In [ ]:
EVAL_SYSTEM_OPENENDED = """You are a rigorous scientific peer reviewer evaluating LLM responses on PhD journal-club level open-ended synthesis questions. These questions have no single correct answer — they require reasoning beyond the source text toward implications, tensions, limitations, or falsifiable hypotheses.

INPUT TYPES:
- [TRACE]: reasoning block present. All criteria start at 10, deduct for failures.
- [FINAL ANSWER ONLY]: no trace. reasoning_quality and intellectual_depth start at 5, max deductions -1.5 each. reasoning_traceability assigned directly from structural tier (no deductions, no bonuses).

TASKS: Score each criterion 0-10. overall_score = mean of three, rounded to 2 decimals. One-sentence summary.

---

CRITERION 1 — reasoning_quality
Does the response build an explicit causal model (named nodes + signed directed edges) from which the conclusion is derived, not merely stated alongside?
Requirements: (a) variables named before reasoning, (b) directed signed relationships with mechanism, (c) competing pathways resolved, (d) conclusion forced by graph structure.
Bulleted lists, numbered mechanisms, and headers are NOT causal models even with arrow notation.

Deductions [TRACE]:
- No graph structure and no directed causal relationships: -3
- Graph present but conclusion not derived from traversal: -1.5
- Edges unsigned or missing mechanism: -0.5 each type, max -1.5
- Competing pathways unresolved: -0.5
- Contradictory edges without regime specification: -0.5
- Mechanism named without causal pathway: -0.5 each, max -1.5
- Hedging language (\"maybe,\" \"perhaps\"): -0.5 each, max -1
- Think block meanders, not reflected in final answer: -1

Bonus [TRACE]: two competing pathways genuinely considered, one resolved through reasoning, reflected in conclusion: +1. This bonus MUST be applied whenever these conditions are met — it is not discretionary.

---

CRITERION 2 — intellectual_depth
Does the response derive WHY a pattern holds rather than naming it?
Naming a phenomenon (\"viscoelastic overshoot,\" \"percolation transition\") without explaining the causal pathway for why it applies here IS a failure.

Deductions [TRACE]:
- Named phenomenon without causal pathway: -0.5 each, max -1
- No non-obvious patterns identified: -1.5
  EXEMPTION: do NOT apply this deduction if the response constructs an explicit causal graph with threshold effects, competing pathways, or feedback loops present as edges. The graph structure itself constitutes pattern identification.
- System-level emergent behavior absent: -0.5
  EXEMPTION: do NOT apply this deduction if competing causal pathways are explicitly modeled and resolved in the reasoning. Pathway resolution demonstrates emergent reasoning.
- Conclusion restates earlier observations: -0.5
- Counterfactuals mentioned but not pursued: -0.5
- Untraced claims in conclusion: -0.5

Bonuses [TRACE]:
- At least one threshold/feedback/competing pathway partially derived: +1 (not discretionary).
- Explicit causal graph with ≥2 resolved competing edges: +0.5 (not discretionary).

---

CRITERION 3 — reasoning_traceability
Can a reader reconstruct entities → relationships → patterns → conclusion without ambiguity? Does reasoning force the conclusion rather than merely precede it?

Structural tiers (apply one):
- Explicit graph, named nodes + signed edges before synthesis, conclusion from traversal: [TRACE] -1 | [FINAL] assign 4
- Exploratory think block, competing hypotheses resolved, reflected in conclusion: [TRACE] -3 | [FINAL] n/a
- Organized prose, parallel sections independent not cumulative: [TRACE] -5 | [FINAL] assign 3
- Decorative think block, no competing hypotheses generated or resolved: [TRACE] -7 | [FINAL] n/a
- Untraced or incoherent, conclusion pre-writable: [TRACE] -7 | [FINAL] assign 2

Additional deductions [TRACE]: contradictory edges unresolved -0.5, backtracking -0.5, repetition -0.5, untraced claims -0.5.

---

RULES: Formatting is not evidence of reasoning. Score logical structure only. Empty response: all scores = 0.

RESPOND WITH ONLY a valid JSON object:
"""

EVAL_USER_OPENENDED = """Question: {question}
Model response: {input_type}
{model_answer}"""

## 3. The judge call (`evaluate_row`)

In [ ]:
def evaluate_row(row: pd.Series) -> Dict[str, Any]:
    """Score one inference row.

    Branches on whether `thinking` is present:
      • `thinking` set    → [TRACE] mode, scored on the full reasoning block
      • `thinking` empty  → [FINAL ANSWER ONLY] mode, scored on `raw_output`
    """
    thinking = str(row.get("thinking") or "").strip()
    if thinking:
        input_type, answer = "[TRACE]", thinking
    else:
        input_type, answer = "[FINAL ANSWER ONLY]", str(row.get("raw_output") or "EMPTY").strip()

    user_msg = EVAL_USER_OPENENDED.format(
        question=str(row["question"]),
        input_type=input_type,
        model_answer=answer,
    )

    try:
        resp = client.messages.create(
            model=MODEL,
            max_tokens=2000,
            system=EVAL_SYSTEM_OPENENDED,
            messages=[{"role": "user", "content": user_msg}],
        )
        raw = resp.content[0].text.strip()
        if not raw:
            raise ValueError("Empty response text")
        result = EvalResult(**json.loads(raw))
        return {
            "reasoning_quality":      result.reasoning_quality,
            "intellectual_depth":     result.intellectual_depth,
            "reasoning_traceability": result.reasoning_traceability,
            "overall_score":          result.overall_score,
            "summary":                result.summary,
            "input_type":             input_type,
            "error":                  result.error,
        }
    except Exception as e:
        print(f"[evaluate_row ERROR] id={row.get('id')} input_type={input_type}: {e}")
        return {k: 0.0 for k in ["reasoning_quality","intellectual_depth","reasoning_traceability","overall_score"]} | {
            "summary": "", "input_type": input_type, "error": str(e),
        }

## 4. Per-model helpers

`load_inference_runs(name, has_phases=...)` loads a single inference JSONL from `data/results/{name}.jsonl` (produced by [`scripts/eval_*.py`](../scripts/) against the full 100-question benchmark) and, for Graph-PRefLexOR variants, rebuilds the `<brainstorm>...<synthesis>` `thinking` column from the five phase columns.

`run_eval(df, out_name)` calls `evaluate_row` for every row, attaches the four score columns + summary, prints non-zero averages, and writes the scored dataframe to `data/results/{out_name}`.

In [ ]:
GRAPH_PHASES = ["brainstorm", "graph", "graph_json", "patterns", "synthesis"]

def build_thinking_block(df: pd.DataFrame) -> pd.Series:
    """Reassemble the structured `<think>` block from the five phase columns."""
    return sum(
        (f"<{p}>\n" + df[p].astype(str) + f"\n</{p}>\n\n" for p in GRAPH_PHASES),
        start=pd.Series([""] * len(df), index=df.index),
    )

def load_inference_runs(name: str, has_phases: bool = False,
                        dedupe_on: str = "question") -> pd.DataFrame:
    """Load one inference JSONL from RESULTS_DIR (no sharding — all 100 questions are
    in a single file per model). If `has_phases`, rebuild the `thinking` column from
    the five Graph-PRefLexOR phase columns.
    """
    path = RESULTS_DIR / f"{name}.jsonl"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing inference file: {path}\n"
            f"Produced by scripts/eval_*.py. If you only have the previously\n"
            f"evaluated {name.replace('_results','')}_data_eval_all.jsonl, skip §5 and jump to §6."
        )
    df = pd.read_json(path, lines=True)
    if has_phases:
        df["thinking"] = build_thinking_block(df)
    df = df.drop_duplicates(subset=[dedupe_on], keep="last").reset_index(drop=True)
    print(f"{name}: {len(df)} rows")
    return df

def run_eval(df: pd.DataFrame, out_name: str) -> pd.DataFrame:
    """Run Claude-as-judge over every row; attach scores, save JSONL, print averages."""
    rows = [evaluate_row(df.iloc[i]) for i in tqdm(range(len(df)), desc=out_name)]
    for col in ("reasoning_quality", "intellectual_depth", "reasoning_traceability",
                "overall_score", "summary"):
        df[col + ("_score" if col != "overall_score" and col != "summary" else "")] = [r[col] for r in rows]

    nonzero = lambda c: [v for v in df[c] if v != 0]
    for col, label in [
        ("reasoning_quality_score",      "Reasoning quality"),
        ("intellectual_depth_score",     "Intellectual depth"),
        ("reasoning_traceability_score", "Reasoning traceability"),
        ("overall_score",                "Overall"),
    ]:
        nz = nonzero(col)
        avg = f"{sum(nz)/len(nz):.2f}" if nz else "N/A"
        print(f"  Avg {label:25s} (n={len(nz):3d}/{len(df)}): {avg}")

    out_path = RESULTS_DIR / out_name
    df.to_json(out_path, orient="records", lines=True)
    print(f"  Saved → {out_path.name}")
    return df

## 5. Run evaluation per model

Each cell below loads one model's inference shards and runs the Claude judge over them. Cells are independent so individual models can be re-run without touching the others. Outputs are written to `data/results/<model>_data_eval_all.jsonl`.

### 5.1 Graph-PRefLexOR-8B

In [ ]:
graph_8b   = load_inference_runs("graph_8b_results",   has_phases=True)
graph_8b   = run_eval(graph_8b,   "graph_8b_data_eval_all.jsonl")

### 5.2 Qwen3-8B (thinking on)

In [ ]:
qwen_8b    = load_inference_runs("qwen_8b_results",    has_phases=False)
qwen_8b    = run_eval(qwen_8b,    "qwen_8b_data_eval_all.jsonl")

### 5.3 Qwen3-8B (no-thinking ablation)

In [ ]:
qwen_8b_nt = load_inference_runs("qwen_8b_no_thinking_results", has_phases=False)
qwen_8b_nt = run_eval(qwen_8b_nt, "qwen_8b_no_thinking_data_eval_all.jsonl")

### 5.4 Graph-PRefLexOR-3B

In [ ]:
graph_3b   = load_inference_runs("graph_3b_results",   has_phases=True)
graph_3b   = run_eval(graph_3b,   "graph_3b_data_eval_all.jsonl")

### 5.5 Llama-3.2-3B-Instruct

In [ ]:
llama_3b   = load_inference_runs("llama_3b_results",   has_phases=False)
llama_3b   = run_eval(llama_3b,   "llama_3b_data_eval_all.jsonl")

### 5.6 Graph-PRefLexOR-1.7B

In [ ]:
graph_1_7b = load_inference_runs("graph_1_7b_results", has_phases=True)
graph_1_7b = run_eval(graph_1_7b, "graph_1_7b_data_eval_all.jsonl")

### 5.7 Qwen3-1.7B (thinking on)

In [ ]:
qwen_1_7b  = load_inference_runs("qwen_1_7b_results",  has_phases=False)
qwen_1_7b  = run_eval(qwen_1_7b,  "qwen_1_7b_data_eval_all.jsonl")

### 5.8 Qwen3-1.7B (no-thinking ablation)

In [ ]:
qwen_1_7b_nt = load_inference_runs("qwen_1_7b_no_thinking_results", has_phases=False)
qwen_1_7b_nt = run_eval(qwen_1_7b_nt, "qwen_1_7b_data_no_thinking_eval_all.jsonl")

## 6. Figure 1 — grouped-bar plots

Reproduces the four panels in Figure 1 of the paper. Panels (a)–(c) compare each Graph-PRefLexOR variant against its base model (and no-thinking ablation where applicable); panel (d) compares the three Graph-PRefLexOR scales head-to-head.

In [ ]:
METRICS = {
    "Reasoning\nQuality":      "reasoning_quality_score",
    "Intellectual\nDepth":     "intellectual_depth_score",
    "Reasoning\nTraceability": "reasoning_traceability_score",
    "Overall":                  "overall_score",
}
BAR_COLORS = ["#4C72B0", "#DD8452", "#55A868"]

def mean_scores(eval_file: str) -> Dict[str, float]:
    """Load an eval JSONL and return the mean of each of the four metrics."""
    df = pd.read_json(RESULTS_DIR / eval_file, lines=True)
    return {label: df[col].mean() for label, col in METRICS.items()}

def plot_panel(model_scores: List[tuple], title: str) -> None:
    """Render one grouped-bar panel. `model_scores` is a list of (label, scores_dict) pairs."""
    metrics = list(METRICS.keys())
    x = np.arange(len(metrics)) * 2
    width = 0.35
    offsets = np.linspace(-width, width, len(model_scores)) if len(model_scores) > 1 else [0]

    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    all_bars = []
    for off, (label, scores), color in zip(offsets, model_scores, BAR_COLORS):
        bars = ax.bar(x + off, [scores[m] for m in metrics], width,
                      label=label, color=color, alpha=0.85,
                      edgecolor="black", linewidth=0.8)
        all_bars.extend(bars)

    for bar in all_bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"{bar.get_height():.2f}",
                ha="center", va="bottom", fontsize=9, fontweight="bold")

    plt.rcParams.update({"font.size": 20})
    ax.set_ylabel("Score", fontsize=20)
    ax.set_title(title, fontsize=20, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=20)
    ax.set_ylim(0, 11)
    ax.axhline(y=5, color="gray", linestyle="--", alpha=0.4, linewidth=0.8)
    ax.legend(fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

### Figure 1a — 8B models

In [ ]:
plot_panel(
    [
        ("Graph-PRefLexOR-8B",  mean_scores("graph_8b_data_eval_all.jsonl")),
        ("Qwen-8B",             mean_scores("qwen_8b_data_eval_all.jsonl")),
        ("Qwen-8B-No-Thinking", mean_scores("qwen_8b_no_thinking_data_eval_all.jsonl")),
    ],
    title="Reasoning Trace Evaluation: 8B Models",
)

### Figure 1b — 3B models

In [ ]:
plot_panel(
    [
        ("Graph-PRefLexOR-3B", mean_scores("graph_3b_data_eval_all.jsonl")),
        ("Llama-3B",           mean_scores("llama_3b_data_eval_all.jsonl")),
    ],
    title="Reasoning Trace Evaluation: 3B Models",
)

### Figure 1c — 1.7B models

In [ ]:
plot_panel(
    [
        ("Graph-PRefLexOR-1.7B",  mean_scores("graph_1_7b_data_eval_all.jsonl")),
        ("Qwen-1.7B",             mean_scores("qwen_1_7b_data_eval_all.jsonl")),
        ("Qwen-1.7B-No-Thinking", mean_scores("qwen_1_7b_no_thinking_data_eval_all.jsonl")),
    ],
    title="Reasoning Trace Evaluation: 1.7B Models",
)

### Figure 1d — Graph-PRefLexOR scaling

In [ ]:
plot_panel(
    [
        ("Graph-PRefLexOR-8B",   mean_scores("graph_8b_data_eval_all.jsonl")),
        ("Graph-PRefLexOR-3B",   mean_scores("graph_3b_data_eval_all.jsonl")),
        ("Graph-PRefLexOR-1.7B", mean_scores("graph_1_7b_data_eval_all.jsonl")),
    ],
    title="Reasoning Trace Evaluation: Graph-PRefLexOR",
)